# **Análisis de grandes volúmenes de datos (TC4034)**

## Maestría en Inteligencia Artificial Aplicada
### Ivan Olmos Pineda | Perla A. García Aguirre
### Tecnológico de Monterrey
## **Proyecto | Base de Datos de Big Data**

---

### Team 8

- Michelle Andrea Arceo Solano — A01625268
- Jacobo Daniel Salazar García — A01796997
- Ariel Antonio Alvarez Monroy — A01796838
- Omar Aguilar Macedo - A0179707

# Descripción del proyecto

## NYC Taxi Trip Records

## Contexto del Dataset

El dataset **NYC Taxi Trip Records** contiene información detallada sobre los viajes realizados por taxis en la ciudad de Nueva York. Cada registro representa un viaje individual e incluye variables relacionadas con tiempo, ubicación, distancia y costo del servicio.

Este dataset es ampliamente utilizado en proyectos de Big Data debido a su gran volumen, riqueza de variables y aplicabilidad en análisis de patrones urbanos, predicción y segmentación.

| Atributo         | Detalle |
|------------------|---------|
| **Nombre**       | NYC Yellow Taxi Trip Data |
| **Origen**       | Taxi & Limousine Commission (TLC) — NYC Government |
| **Fuente oficial**| https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page |
| **Diccionario de datos**| https://www.nyc.gov/assets/tlc/downloads/pdf/data_dictionary_trip_records_yellow.pdf |
| **Tabla de zonas (lookup)**| https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv |
| **Período a analizar**      | Enero 2023 — Marzo 2026 |
| **Tamaño total** | ~2.5 GB (139,279,754 registros, 24 columnas) |
| **Tamaño por archivo mensual** | ~45MB - 70MB (Parquet)   |
| **Formato**      | parquet |




## Tipos de Datos Disponibles

El dataset está dividido en diferentes categorías según el tipo de servicio:

- Yellow Taxi Trips  
- Green Taxi Trips  
- For-Hire Vehicle (FHV) Trips  
- High Volume FHV (Uber, Lyft, etc.)

Para este proyecto se seleccionó: **Yellow Taxi Trips**

## Formato y Acceso

Los datos están disponibles en formato `Parquet`

Este formato resulta eficiente, ya que se puede comprimir, permite lectura de columnas, y tiene un mejor rendimiento al usarse con PySpark

Ejemplo de archivo:
- https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet



# Preparación del entorno

In [1]:
# @title Dataset Imports
import os
import requests
from pathlib import Path

In [2]:
# @title Conectar a google drive
from google.colab import drive
drive.mount('/content/drive', force_remount = True)

Mounted at /content/drive


In [3]:
#@title Validación de Java
!java --version
!which java

openjdk 17.0.18 2026-01-20
OpenJDK Runtime Environment (build 17.0.18+8-Ubuntu-122.04.1)
OpenJDK 64-Bit Server VM (build 17.0.18+8-Ubuntu-122.04.1, mixed mode, sharing)
/usr/bin/java


In [4]:
#@title Agregamos JAVA_HOME

# Solo para linux :S, por lo que funciona en colab
JAVA_HOME=!java -XshowSettings:properties -version 2>&1 > /dev/null | grep 'java.home' | awk '{print $NF}'
JAVA_HOME=JAVA_HOME[0] # El comando anterior regresa un arreglo
%env JAVA_HOME=$JAVA_HOME

env: JAVA_HOME=/usr/lib/jvm/java-17-openjdk-amd64


In [5]:
#@title Configuración e instalación de PySpark
!pip install pyspark
!pip install findspark

In [6]:
#@title PySpark imports
import findspark
from pyspark.sql import SparkSession

from pyspark.sql import functions as F
from pyspark.sql import types as T
from functools import reduce
import glob

# Para cargar algunos datasets estadísticos simples
import pandas as pd

# Para crear graficos simples
import matplotlib.pyplot as plt

# Descarga y lectura del dataset

In [7]:
# @title Descargar Dataset

# drive_dir = "/content/drive/MyDrive/TC4034_BigData/proyecto/taxi/yellow"

drive_dir = '/content/drive/MyDrive/TC4034.10 - Análisis de grandes volúmenes de datos (Gpo 10) | Equipo 8/raw/taxi/yellow'

In [8]:
#@title Leer el dataset

spark = (
    SparkSession.builder
    .appName("NYC Yellow Taxi")
    .getOrCreate()
)

# clean folder contains the parquet with the dataset with modifications
df = spark.read.parquet(f"{drive_dir}/clean")

In [9]:
df.printSchema()

root
 |-- VendorID: byte (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: byte (nullable = true)
 |-- trip_distance: float (nullable = true)
 |-- RatecodeID: byte (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: short (nullable = true)
 |-- DOLocationID: short (nullable = true)
 |-- payment_type: byte (nullable = true)
 |-- fare_amount: float (nullable = true)
 |-- extra: float (nullable = true)
 |-- mta_tax: float (nullable = true)
 |-- tip_amount: float (nullable = true)
 |-- tolls_amount: float (nullable = true)
 |-- improvement_surcharge: float (nullable = true)
 |-- total_amount: float (nullable = true)
 |-- congestion_surcharge: float (nullable = true)
 |-- Airport_fee: float (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)
 |-- pickup_year: short (nullable = true)
 |-- pickup_month: byte (nullable = true)
 |-- pick

In [10]:
#@title Muestra 10 registros
df.limit(10).toPandas()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,pickup_year,pickup_month,pickup_hour,trip_duration_minutes
0,1,2023-05-01 00:33:13,2023-05-01 00:53:01,0,7.80,1,N,138,43,1,...,0.00,1.0,51.650002,0.0,1.75,NaN,2023,5,0,19.8
1,1,2023-05-01 00:42:49,2023-05-01 01:11:18,2,8.10,1,N,138,262,1,...,0.00,1.0,57.150002,2.5,1.75,NaN,2023,5,0,28.5
2,1,2023-05-01 00:56:34,2023-05-01 01:13:39,2,9.10,1,N,138,141,1,...,6.55,1.0,64.199997,2.5,1.75,NaN,2023,5,0,17.1
3,2,2023-05-01 00:00:52,2023-05-01 00:20:12,1,8.21,1,N,138,140,1,...,0.00,1.0,47.090000,2.5,1.75,NaN,2023,5,0,19.3
4,1,2023-05-01 00:05:50,2023-05-01 00:19:41,0,7.90,1,N,138,263,1,...,6.55,1.0,59.150002,2.5,1.75,NaN,2023,5,0,13.9
5,1,2023-05-01 00:42:54,2023-05-01 01:04:49,0,10.40,1,N,138,246,1,...,6.55,1.0,69.000000,2.5,1.75,NaN,2023,5,0,21.9
6,2,2023-05-01 00:50:34,2023-05-01 01:12:09,1,9.05,1,N,138,116,1,...,6.55,1.0,64.559998,0.0,1.75,NaN,2023,5,0,21.6
7,1,2023-05-01 00:13:58,2023-05-01 00:18:10,1,0.70,1,N,161,48,1,...,0.00,1.0,14.350000,2.5,0.00,NaN,2023,5,0,4.2
8,2,2023-04-30 23:48:31,2023-04-30 23:57:35,1,2.38,1,N,249,231,2,...,0.00,1.0,17.100000,2.5,0.00,NaN,2023,4,23,9.1
9,2,2023-05-01 00:28:47,2023-05-01 00:39:33,1,2.92,1,N,114,230,2,...,0.00,1.0,19.900000,2.5,0.00,NaN,2023,5,0,10.8


In [11]:
#@title Dimensiones generales

total_records = df.count()
total_columns = len(df.columns)
size_in_bytes = df._jdf.queryExecution().optimizedPlan().stats().sizeInBytes()
print(f"Número de registros: {total_records:,}")
print(f"Información de columnas: {total_columns}")
print(f"Tamaño aproximado: ~{(size_in_bytes / 1024 / 1024 / 1024):.02f} GB")


Número de registros: 139,279,754
Información de columnas: 24
Tamaño aproximado: ~2.87 GB


## Caracterización de la Población

In [12]:
#@title (Descargar archivo auxiliar de zonas para analisis)

import os
import requests

zones_url = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"
zones_path = f"{drive_dir}/taxi_zone_lookup.csv"

if not os.path.exists(zones_path):
    response = requests.get(zones_url)
    response.raise_for_status()

    with open(zones_path, "wb") as f:
        f.write(response.content)

    print("Archivo descargado correctamente:")
    print(zones_path)
else:
    print("El archivo ya existe:")
    print(zones_path)

El archivo ya existe:
/content/drive/MyDrive/TC4034_BigData/proyecto/taxi/yellow/taxi_zone_lookup.csv


### Ejemplo 01

In [13]:
df_partitioned = (
    df
    .withColumn(
        "pickup_hour_bucket",
        F.when((F.col("pickup_hour") >= 0) & (F.col("pickup_hour") <= 5), "madrugada")
         .when((F.col("pickup_hour") >= 6) & (F.col("pickup_hour") <= 11), "mañana")
         .when((F.col("pickup_hour") >= 12) & (F.col("pickup_hour") <= 17), "tarde")
         .otherwise("noche")
    )
    .withColumn(
        "trip_duration_bucket",
        F.when(F.col("trip_duration_minutes") <= 10, "corto")
         .when(F.col("trip_duration_minutes") <= 30, "medio")
         .otherwise("largo")
    )
)

In [14]:
partitions = (
    df_partitioned
    .groupBy("pickup_hour_bucket", "trip_duration_bucket")
    .count()
    .orderBy("pickup_hour_bucket", "trip_duration_bucket")
)

partitions.show()

+------------------+--------------------+--------+
|pickup_hour_bucket|trip_duration_bucket|   count|
+------------------+--------------------+--------+
|         madrugada|               corto| 4933567|
|         madrugada|               largo|  762978|
|         madrugada|               medio| 6125615|
|            mañana|               corto|11290540|
|            mañana|               largo| 3719080|
|            mañana|               medio|14911773|
|             noche|               corto|17608286|
|             noche|               largo| 4560621|
|             noche|               medio|26161083|
|             tarde|               corto|16185838|
|             tarde|               largo| 7934605|
|             tarde|               medio|25085768|
+------------------+--------------------+--------+



In [15]:
total_rows = df_partitioned.count()

partition_stats = (
    partitions
    .withColumn("probability", F.col("count") / F.lit(total_rows))
)

partition_stats.show()

+------------------+--------------------+--------+--------------------+
|pickup_hour_bucket|trip_duration_bucket|   count|         probability|
+------------------+--------------------+--------+--------------------+
|         madrugada|               corto| 4933567| 0.03542199679646189|
|         madrugada|               largo|  762978|0.005478025183760735|
|         madrugada|               medio| 6125615| 0.04398065637020008|
|            mañana|               corto|11290540| 0.08106375604310731|
|            mañana|               largo| 3719080| 0.02670222981582808|
|            mañana|               medio|14911773| 0.10706346451473485|
|             noche|               corto|17608286| 0.12642387349420506|
|             noche|               largo| 4560621|0.032744321188275506|
|             noche|               medio|26161083| 0.18783119763407968|
|             tarde|               corto|16185838| 0.11621098928707183|
|             tarde|               largo| 7934605| 0.05696883267

In [16]:
sample_morning_short = (
    df_partitioned
    .filter(
        (F.col("pickup_hour_bucket") == "mañana") &
        (F.col("trip_duration_bucket") == "corto")
    )
    .limit(1000)
)

sample_morning_short.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+-----------+------------+-----------+---------------------+------------------+--------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|pickup_year|pickup_month|pickup_hour|trip_duration_minutes|pickup_hour_bucket|trip_duration_bucket|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+--------------------

In [17]:
sample_fraction = 0.01  # 1% de cada partición

sample = (
    df_partitioned
    .withColumn(
        "partition_key",
        F.concat_ws(
            "_",
            F.col("pickup_hour_bucket"),
            F.col("trip_duration_bucket")
        )
    )
    .sampleBy(
        "partition_key",
        fractions={
            "madrugada_corto": sample_fraction,
            "madrugada_medio": sample_fraction,
            "madrugada_largo": sample_fraction,
            "mañana_corto": sample_fraction,
            "mañana_medio": sample_fraction,
            "mañana_largo": sample_fraction,
            "tarde_corto": sample_fraction,
            "tarde_medio": sample_fraction,
            "tarde_largo": sample_fraction,
            "noche_corto": sample_fraction,
            "noche_medio": sample_fraction,
            "noche_largo": sample_fraction,
        },
        seed=42
    )
)

In [18]:
sample.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+-----------+------------+-----------+---------------------+------------------+--------------------+---------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|pickup_year|pickup_month|pickup_hour|trip_duration_minutes|pickup_hour_bucket|trip_duration_bucket|  partition_key|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+-

### Ejemplo 02

#### Caracterización estadística de las variables de particionamiento

Las dos variables utilizadas para particionar el dataset fueron seleccionadas a partir del análisis exploratorio previo. A continuación se documenta su dominio y comportamiento estadístico observado.

| Variable | Dominio | Descripción estadística observada | Comentarios |
|---|---|---|---|
| `hora_segmento` | `pico` / `fuera_pico` | P(pico) ≈ 0.4289 · P(fuera_pico) ≈ 0.5711. Las horas pico se definen como 7-9 h y 16-20 h, franjas con mayor concentración de viajes según el histograma de `pickup_hour`. | La distribución es consistente con patrones de movilidad urbana reportados por la TLC. Los viajes en hora pico tienden a tener mayor tarifa promedio y menor duración por distancia recorrida. |
| `zona_origen` | `manhattan` / `no_manhattan` | P(manhattan) ≈ 0.8724 · P(no_manhattan) ≈ 0.1276. Manhattan concentra la gran mayoría de los inicios de viaje de Yellow Taxi. Las zonas fuera de Manhattan incluyen principalmente aeropuertos (JFK, LGA), Queens, Brooklyn y el Bronx. | El alto desbalance entre clases justifica el uso de muestreo estratificado proporcional para garantizar representación de viajes no-Manhattan en los conjuntos de entrenamiento y prueba. |

> Los valores de probabilidad fueron calculados empíricamente sobre el dataset completo (ver celda "Derivación empírica de probabilidades").

La combinación de estas dos variables genera **cuatro particiones** exhaustivas y mutuamente excluyentes:

| Partición | `hora_segmento` | `zona_origen` | Prob. combinada estimada | Prob. observada (Cell 61) |
|---|---|---|---|---|
| P1 | `pico` | `manhattan` | 0.4289 x 0.8724 ≈ **0.3742** | 0.3761 |
| P2 | `pico` | `no_manhattan` | 0.4289 x 0.1276 ≈ **0.0547** | 0.0492 |
| P3 | `fuera_pico` | `manhattan` | 0.5711 x 0.8724 ≈ **0.4982** | 0.5006 |
| P4 | `fuera_pico` | `no_manhattan` | 0.5711 x 0.1276 ≈ **0.0729** | 0.0740 |

> La leve diferencia entre probabilidades estimadas y observadas se debe a que el filtro de calidad de datos (fare > 0, distance > 0, duration ≤ 180 min) elimina registros de forma no uniforme entre particiones.

In [19]:
# Recargar zonas sobre la nueva sesión
zones = spark.read.csv(zones_path, header=True, inferSchema=True)

zones_pu = zones.select(
    F.col("LocationID").alias("_PUKey"),
    F.col("Borough").alias("borough_origen")
)

In [20]:
#@title Carga de datos y filtrado de registros válidos

df_raw = (
    df
    .withColumn("pickup_year",  F.year("tpep_pickup_datetime"))
    .withColumn("pickup_month", F.month("tpep_pickup_datetime"))
    .withColumn("pickup_hour",  F.hour("tpep_pickup_datetime"))
    .withColumn("trip_duration_minutes",
        (F.unix_timestamp("tpep_dropoff_datetime") -
          F.unix_timestamp("tpep_pickup_datetime")) / 60.0)
)
df_part = df_raw.filter(
    (F.col("pickup_year").between(2023, 2026)) &
    ~(
        (F.col("pickup_year")  == 2026) &
        (F.col("pickup_month") > 3)
    ) &
    (F.col("fare_amount")  > 0) &
    (F.col("trip_distance") > 0) &
    (F.col("trip_duration_minutes") >  0) &
    (F.col("trip_duration_minutes") <= 180)
)
total_registros = df_part.count()
print(f"Fallback completado: {total_registros:,} registros válidos.")

print(f"\nRegistros válidos para particionamiento: {total_registros:,}")
print("\nSchema de df_part:")
df_part.printSchema()

print("\nDistribución por año:")
df_part.groupBy("pickup_year").count().orderBy("pickup_year").show()


Fallback completado: 131,471,242 registros válidos.

Registros válidos para particionamiento: 131,471,242

Schema de df_part:
root
 |-- VendorID: byte (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: byte (nullable = true)
 |-- trip_distance: float (nullable = true)
 |-- RatecodeID: byte (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: short (nullable = true)
 |-- DOLocationID: short (nullable = true)
 |-- payment_type: byte (nullable = true)
 |-- fare_amount: float (nullable = true)
 |-- extra: float (nullable = true)
 |-- mta_tax: float (nullable = true)
 |-- tip_amount: float (nullable = true)
 |-- tolls_amount: float (nullable = true)
 |-- improvement_surcharge: float (nullable = true)
 |-- total_amount: float (nullable = true)
 |-- congestion_surcharge: float (nullable = true)
 |-- Airport_fee: float (nullable = true)
 |-- cbd_congesti

In [21]:
#@title Agregar variables de caracterización al DataFrame

# ── Variable A: hora_segmento ──────────────────────────────────────────────
df_part = df_part.withColumn(
    "hora_segmento",
    F.when(
        ((F.col("pickup_hour") >= 7)  & (F.col("pickup_hour") <= 9)) |
        ((F.col("pickup_hour") >= 16) & (F.col("pickup_hour") <= 20)),
        "pico"
    ).otherwise("fuera_pico")
)

# ── Variable B: zona_origen ─────────────────────────────────────────────────
# broadcast() porque zones tiene ~265 filas — evita shuffle del DataFrame grande.
zones_pu = zones.select(
    F.col("LocationID").alias("_PUKey"),
    F.col("Borough").alias("borough_origen")
)

df_part = (
    df_part
    .join(F.broadcast(zones_pu), df_part.PULocationID == F.col("_PUKey"), "left")
    .drop("_PUKey")
)

df_part = df_part.withColumn(
    "zona_origen",
    F.when(F.col("borough_origen") == "Manhattan", "manhattan")
    .otherwise("no_manhattan")
)

# ── Verificar variables agregadas ──────────────────────────────────────────
# La materialización a Parquet se omite: en modo local de Colab, escribir
# ~130 M filas requiere que el driver JVM procese todo el dataset en memoria,
# provocando un OOM que mata el SparkContext.
# El plan lógico resultante (leer PART_DIR → dos withColumn → broadcast join
# de 265 filas) es suficientemente ligero para las operaciones de
# particionamiento y muestreo que siguen, como confirman sus salidas previas.

print("Variables de caracterización agregadas al plan lógico de df_part.")
df_part.printSchema()

print("\nMuestra de verificación (3 filas):")
df_part.select(
    "pickup_hour", "hora_segmento",
    "borough_origen", "zona_origen"
).show(3, truncate=False)


Variables de caracterización agregadas al plan lógico de df_part.
root
 |-- VendorID: byte (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: byte (nullable = true)
 |-- trip_distance: float (nullable = true)
 |-- RatecodeID: byte (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: short (nullable = true)
 |-- DOLocationID: short (nullable = true)
 |-- payment_type: byte (nullable = true)
 |-- fare_amount: float (nullable = true)
 |-- extra: float (nullable = true)
 |-- mta_tax: float (nullable = true)
 |-- tip_amount: float (nullable = true)
 |-- tolls_amount: float (nullable = true)
 |-- improvement_surcharge: float (nullable = true)
 |-- total_amount: float (nullable = true)
 |-- congestion_surcharge: float (nullable = true)
 |-- Airport_fee: float (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)
 |-- pickup_year: integer (

In [22]:
#@title Probabilidades observadas de cada partición

# count() sobre el Parquet materializado, no sobre los 39 archivos originales
total_registros = df_part.count()
print(f"Registros válidos para particionamiento: {total_registros:,}")

partition_stats = (
    df_part
    .groupBy("hora_segmento", "zona_origen")
    .count()
    .withColumn(
        "probabilidad",
        F.round(F.col("count") / F.lit(total_registros), 4)
    )
    .withColumn(
        "particion",
        F.when(
            (F.col("hora_segmento") == "pico") & (F.col("zona_origen") == "manhattan"),
            "P1"
        ).when(
            (F.col("hora_segmento") == "pico") & (F.col("zona_origen") == "no_manhattan"),
            "P2"
        ).when(
            (F.col("hora_segmento") == "fuera_pico") & (F.col("zona_origen") == "manhattan"),
            "P3"
        ).otherwise("P4")
    )
    .select("particion", "hora_segmento", "zona_origen", "count", "probabilidad")
    .orderBy("particion")
)

partition_stats.show(truncate=False)
print(f"Suma de probabilidades: {partition_stats.agg(F.sum('probabilidad')).collect()[0][0]:.4f}")

Registros válidos para particionamiento: 131,471,242
+---------+-------------+------------+--------+------------+
|particion|hora_segmento|zona_origen |count   |probabilidad|
+---------+-------------+------------+--------+------------+
|P1       |pico         |manhattan   |49447865|0.3761      |
|P2       |pico         |no_manhattan|6469023 |0.0492      |
|P3       |fuera_pico   |manhattan   |65820151|0.5006      |
|P4       |fuera_pico   |no_manhattan|9734203 |0.074       |
+---------+-------------+------------+--------+------------+

Suma de probabilidades: 0.9999


In [23]:
#@title Partición P1 — hora_segmento: pico | zona_origen: manhattan

# Regla: pickup_hour IN [7,8,9,16,17,18,19,20] AND borough_origen = 'Manhattan'
# Perfil esperado: viajes cortos y medios, tarifa elevada, uso comercial intenso.

part_1 = df_part.filter(
    (F.col("hora_segmento") == "pico") &
    (F.col("zona_origen")   == "manhattan")
)

n_p1 = part_1.count()
print(f"P1 | Registros: {n_p1:,}")

part_1.select(
    "tpep_pickup_datetime", "pickup_hour", "hora_segmento",
    "borough_origen", "zona_origen",
    "trip_distance", "fare_amount", "total_amount"
).show(5, truncate=False)


P1 | Registros: 49,447,865
+--------------------+-----------+-------------+--------------+-----------+-------------+-----------+------------+
|tpep_pickup_datetime|pickup_hour|hora_segmento|borough_origen|zona_origen|trip_distance|fare_amount|total_amount|
+--------------------+-----------+-------------+--------------+-----------+-------------+-----------+------------+
|2023-05-01 08:44:53 |8          |pico         |Manhattan     |manhattan  |1.67         |9.3        |17.16       |
|2023-05-01 08:52:33 |8          |pico         |Manhattan     |manhattan  |2.39         |12.1       |17.1        |
|2023-05-01 09:12:17 |9          |pico         |Manhattan     |manhattan  |1.18         |7.2        |13.44       |
|2023-05-01 09:46:37 |9          |pico         |Manhattan     |manhattan  |2.15         |11.4       |18.48       |
|2023-05-01 09:59:30 |9          |pico         |Manhattan     |manhattan  |2.54         |13.5       |21.88       |
+--------------------+-----------+-------------+-----

In [24]:
#@title Partición P2 — hora_segmento: pico | zona_origen: no_manhattan

# Regla: pickup_hour IN [7,8,9,16,17,18,19,20] AND borough_origen != 'Manhattan'
# Perfil esperado: viajes más largos con origen en aeropuertos, Queens o Brooklyn.

part_2 = df_part.filter(
    (F.col("hora_segmento") == "pico") &
    (F.col("zona_origen")   == "no_manhattan")
)

n_p2 = part_2.count()
print(f"P2 | Registros: {n_p2:,}")

part_2.select(
    "tpep_pickup_datetime", "pickup_hour", "hora_segmento",
    "borough_origen", "zona_origen",
    "trip_distance", "fare_amount", "total_amount"
).show(5, truncate=False)


P2 | Registros: 6,469,023
+--------------------+-----------+-------------+--------------+------------+-------------+-----------+------------+
|tpep_pickup_datetime|pickup_hour|hora_segmento|borough_origen|zona_origen |trip_distance|fare_amount|total_amount|
+--------------------+-----------+-------------+--------------+------------+-------------+-----------+------------+
|2023-05-01 07:17:48 |7          |pico         |Queens        |no_manhattan|20.1         |70.0       |83.3        |
|2023-05-01 07:36:26 |7          |pico         |Queens        |no_manhattan|22.2         |70.0       |98.75       |
|2023-05-01 07:37:12 |7          |pico         |Queens        |no_manhattan|19.9         |70.0       |82.3        |
|2023-05-01 07:51:10 |7          |pico         |Queens        |no_manhattan|18.73        |70.0       |98.41       |
|2023-05-01 07:24:49 |7          |pico         |Queens        |no_manhattan|21.59        |70.0       |98.76       |
+--------------------+-----------+------------

In [25]:
#@title Partición P3 — hora_segmento: fuera_pico | zona_origen: manhattan

# Regla: pickup_hour NOT IN [7,8,9,16,17,18,19,20] AND borough_origen = 'Manhattan'
# Perfil esperado: viajes nocturnos, recreativos o de madrugada desde el centro.

part_3 = df_part.filter(
    (F.col("hora_segmento") == "fuera_pico") &
    (F.col("zona_origen")   == "manhattan")
)

n_p3 = part_3.count()
print(f"P3 | Registros: {n_p3:,}")

part_3.select(
    "tpep_pickup_datetime", "pickup_hour", "hora_segmento",
    "borough_origen", "zona_origen",
    "trip_distance", "fare_amount", "total_amount"
).show(5, truncate=False)


P3 | Registros: 65,820,151
+--------------------+-----------+-------------+--------------+-----------+-------------+-----------+------------+
|tpep_pickup_datetime|pickup_hour|hora_segmento|borough_origen|zona_origen|trip_distance|fare_amount|total_amount|
+--------------------+-----------+-------------+--------------+-----------+-------------+-----------+------------+
|2023-05-01 00:13:58 |0          |fuera_pico   |Manhattan     |manhattan  |0.7          |6.5        |14.35       |
|2023-04-30 23:48:31 |23         |fuera_pico   |Manhattan     |manhattan  |2.38         |12.1       |17.1        |
|2023-05-01 00:28:47 |0          |fuera_pico   |Manhattan     |manhattan  |2.92         |14.9       |19.9        |
|2023-05-01 00:55:22 |0          |fuera_pico   |Manhattan     |manhattan  |1.19         |7.2        |14.64       |
|2023-05-01 00:28:12 |0          |fuera_pico   |Manhattan     |manhattan  |2.3          |12.1       |19.1        |
+--------------------+-----------+-------------+-----

In [26]:
#@title Partición P4 — hora_segmento: fuera_pico | zona_origen: no_manhattan

# Regla: pickup_hour NOT IN [7,8,9,16,17,18,19,20] AND borough_origen != 'Manhattan'
# Perfil esperado: viajes aeroportuarios nocturnos, traslados inter-borough.

part_4 = df_part.filter(
    (F.col("hora_segmento") == "fuera_pico") &
    (F.col("zona_origen")   == "no_manhattan")
)

n_p4 = part_4.count()
print(f"P4 | Registros: {n_p4:,}")

part_4.select(
    "tpep_pickup_datetime", "pickup_hour", "hora_segmento",
    "borough_origen", "zona_origen",
    "trip_distance", "fare_amount", "total_amount"
).show(5, truncate=False)


P4 | Registros: 9,734,203
+--------------------+-----------+-------------+--------------+------------+-------------+-----------+------------+
|tpep_pickup_datetime|pickup_hour|hora_segmento|borough_origen|zona_origen |trip_distance|fare_amount|total_amount|
+--------------------+-----------+-------------+--------------+------------+-------------+-----------+------------+
|2023-05-01 00:33:13 |0          |fuera_pico   |Queens        |no_manhattan|7.8          |33.8       |51.65       |
|2023-05-01 00:42:49 |0          |fuera_pico   |Queens        |no_manhattan|8.1          |35.9       |57.15       |
|2023-05-01 00:56:34 |0          |fuera_pico   |Queens        |no_manhattan|9.1          |35.2       |64.2        |
|2023-05-01 00:00:52 |0          |fuera_pico   |Queens        |no_manhattan|8.21         |33.1       |47.09       |
|2023-05-01 00:05:50 |0          |fuera_pico   |Queens        |no_manhattan|7.9          |31.0       |59.15       |
+--------------------+-----------+------------

In [27]:
#@title Resumen y verificación de particiones


partition_map = {
    "P1 (pico + manhattan)":       (part_1, n_p1),
    "P2 (pico + no_manhattan)":    (part_2, n_p2),
    "P3 (fuera_pico + manhattan)": (part_3, n_p3),
    "P4 (fuera_pico + no_manhattan)": (part_4, n_p4),
}

rows = []
for nombre, (_, cnt) in partition_map.items():
    rows.append({
        "Partición": nombre,
        "Registros": cnt,
        "Proporción observada": round(cnt / total_registros, 4)
    })

resumen = pd.DataFrame(rows)
resumen["Prop. acumulada"] = resumen["Proporción observada"].cumsum().round(4)

print(resumen.to_string(index=False))
print(f"\nTotal verificado: {resumen['Registros'].sum():,}")

# Verificar que las 4 particiones son mutuamente excluyentes y exhaustivas
union_count = part_1.count() + part_2.count() + part_3.count() + part_4.count()
assert union_count == total_registros, (
    f"Error: la unión de particiones ({union_count:,}) "
    f"no coincide con el total ({total_registros:,})"
)
print("\nVerificación OK: particiones son exhaustivas y mutuamente excluyentes.")


                     Partición  Registros  Proporción observada  Prop. acumulada
         P1 (pico + manhattan)   49447865                0.3761           0.3761
      P2 (pico + no_manhattan)    6469023                0.0492           0.4253
   P3 (fuera_pico + manhattan)   65820151                0.5006           0.9259
P4 (fuera_pico + no_manhattan)    9734203                0.0740           0.9999

Total verificado: 131,471,242

Verificación OK: particiones son exhaustivas y mutuamente excluyentes.
